# Model Serving & Deployment

Building a capable LLM pipeline is only half the job. The other half is making it reliably available to the rest of the system: the risk dashboard, the compliance portal, the analyst's chat interface. This notebook walks through the full serving stack — a FastAPI service that wraps our LLM pipeline, a multi-stage Docker build that keeps the image lean, a LiteLLM proxy that decouples the API surface from the underlying model provider, and a simple benchmark harness to measure latency and throughput before traffic hits production.

The running example is a **filing analysis service**: a POST endpoint that accepts an SEC filing excerpt and returns a structured JSON response with a one-paragraph summary and a list of key risk factors. We progressively add health checks, graceful shutdown, and request-level logging so the service is ready for a containerized deployment behind a load balancer.

Setup:

In [ ]:
#| echo: false
import os, json, time, asyncio, subprocess, textwrap
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## FastAPI Service

**Why FastAPI?** FastAPI is the standard choice for Python LLM services: it provides automatic request validation via Pydantic, async request handling for non-blocking I/O (critical when each request may wait several seconds for an LLM response), and auto-generated OpenAPI docs. The async model is important because LLM API calls are network-bound; a synchronous server would block its worker thread while waiting for OpenAI, severely limiting concurrency.

<br>

**Service contract.** We define (1) a `FilingRequest` input schema with a `text` field containing the raw filing excerpt and an optional `max_risks` integer, and (2) a `FilingResponse` output schema with a `summary` string and a `risk_factors` list. The LLM is instructed to respond in structured JSON, which FastAPI validates before returning it to the caller. If the LLM response fails validation the service returns a `422 Unprocessable Entity` with details — this is far more useful to a downstream service than a silent empty string.

<br>

**Lifespan context.** FastAPI's `lifespan` parameter replaces the deprecated `on_startup`/`on_shutdown` events. We use it to initialize the `LLMClient` once at startup (avoiding a cold-start penalty on the first request) and log a clean shutdown message. The lifespan pattern also makes it easy to close database connections or flush observability buffers on exit.

The service is defined in `src/main.py`:

```python
# src/main.py
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import openai, os, json, logging
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Pydantic schemas ──────────────────────────────────────────────────────────

class FilingRequest(BaseModel):
    text: str
    max_risks: int = 5

class FilingResponse(BaseModel):
    summary: str
    risk_factors: list[str]

# ── Application state ─────────────────────────────────────────────────────────

class AppState:
    client: openai.AsyncOpenAI = None

state = AppState()

@asynccontextmanager
async def lifespan(app: FastAPI):
    logger.info("Starting up — initialising OpenAI async client")
    state.client = openai.AsyncOpenAI()          # <1>
    yield
    logger.info("Shutting down — flushing client")
    await state.client.close()

app = FastAPI(title="Filing Analyser", lifespan=lifespan)

# ── Health check ──────────────────────────────────────────────────────────────

@app.get("/health")
async def health():
    return {"status": "ok"}

# ── Core endpoint ─────────────────────────────────────────────────────────────

SYSTEM = (
    "You are a financial analyst. Given an SEC filing excerpt, "
    "return a JSON object with fields: "
    "'summary' (one paragraph) and 'risk_factors' (list of short strings, "
    "at most {max_risks})."
)

@app.post("/analyse", response_model=FilingResponse)
async def analyse(req: FilingRequest):           # <2>
    prompt = SYSTEM.format(max_risks=req.max_risks)
    try:
        resp = await state.client.chat.completions.create(
            model=os.getenv("MODEL", "gpt-4o-mini"),
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user",   "content": req.text},
            ],
            response_format={"type": "json_object"},
            temperature=0.0,
        )
        data = json.loads(resp.choices[0].message.content)
        return FilingResponse(**data)            # <3>
    except (json.JSONDecodeError, KeyError) as e:
        raise HTTPException(status_code=422, detail=str(e))
```

1. `openai.AsyncOpenAI()` uses `httpx` under the hood — a single client instance is shared across all requests and manages an HTTP connection pool automatically.
2. FastAPI validates `FilingRequest` before the function body executes; invalid requests are rejected with a `422` before ever reaching the LLM.
3. Constructing `FilingResponse(**data)` validates the LLM's JSON output against the schema. Any missing or wrongly-typed field raises a `ValidationError` caught by the `except` block.

## Docker Build

**Multi-stage builds.** A naive `pip install -r requirements.txt` on top of a full Python base image produces images north of 1 GB. A multi-stage build separates the *build environment* (with compilers, wheel builders, and build tools) from the *runtime environment* (a slim image containing only what the application needs to run). The result is an image typically under 200 MB — faster to push, faster to pull, and with a smaller attack surface.

<br>

**Layer caching discipline.** Docker caches layers in order. We copy `requirements.txt` and install dependencies *before* copying application code so that dependency installation (the slow step) is only re-run when `requirements.txt` changes, not on every code change. This halves CI times once the dependency layer is cached.

<br>

**Non-root user.** Running as root inside a container is a security anti-pattern. We create a dedicated `appuser` and switch to it before the `CMD` so that even if an attacker achieves code execution inside the container, they hold no elevated host privileges.

The `Dockerfile` for the filing analyser service:

```dockerfile
# ── Stage 1: builder ──────────────────────────────────────────────────────────
FROM python:3.13-slim AS builder

WORKDIR /build
COPY requirements.txt .
RUN pip install --upgrade pip \
 && pip install --no-cache-dir --prefix=/install -r requirements.txt  # <1>

# ── Stage 2: runtime ──────────────────────────────────────────────────────────
FROM python:3.13-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PORT=8000

WORKDIR /app
COPY --from=builder /install /usr/local  # <2>
COPY src/ ./src/

RUN adduser --disabled-password --gecos "" appuser \
 && chown -R appuser /app
USER appuser                             # <3>

HEALTHCHECK --interval=30s --timeout=5s --start-period=10s --retries=3 \
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:$PORT/health')"

CMD ["uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000", \
     "--workers", "2", "--timeout-graceful-shutdown", "30"]  # <4>
```

1. `--prefix=/install` installs everything into a directory we can `COPY` wholesale into the runtime stage, leaving the builder image's system Python untouched.
2. The entire `/install` tree (site-packages, scripts) is copied into `/usr/local` in the slim runtime image — no pip, no compiler, no build cache.
3. All subsequent instructions and the final `CMD` run as `appuser`, not root.
4. `--timeout-graceful-shutdown 30` gives in-flight requests up to 30 seconds to complete before uvicorn forcefully terminates workers on `SIGTERM` — essential for Kubernetes rolling deploys.

## LiteLLM Proxy

**The provider abstraction problem.** Application code that calls `openai.chat.completions.create(model="gpt-4o-mini", ...)` is tightly coupled to OpenAI. Switching to Anthropic, a self-hosted Llama model, or a fine-tuned endpoint requires touching every callsite. LiteLLM solves this by exposing a single OpenAI-compatible API surface while routing to any backend based on the `model` string.

<br>

**What the proxy gives us.** Running LiteLLM as a sidecar or dedicated service adds: (1) **model aliasing** — `model: gpt-4o-mini` in code can map to `azure/gpt-4o-mini-prod` in the proxy config without code changes; (2) **fallbacks** — if OpenAI is degraded the proxy automatically retries against Anthropic; (3) **budget controls** — per-team or per-key spend limits enforced at the proxy layer before any token is consumed; (4) **unified logging** — every request logged to a database regardless of backend.

<br>

**Deployment pattern.** In a Kubernetes deployment the proxy runs as a separate `Deployment` with its own `Service`. Application pods send all LLM traffic to `http://litellm-proxy/v1` and never hold provider API keys — keys are stored as Kubernetes `Secret` objects mounted only into the proxy pod.

A minimal LiteLLM proxy configuration covering OpenAI and a self-hosted Ollama endpoint:

```yaml
# litellm_config.yaml
model_list:
  - model_name: gpt-4o-mini          # alias used by application code
    litellm_params:
      model: openai/gpt-4o-mini
      api_key: os.environ/OPENAI_API_KEY

  - model_name: gpt-4o
    litellm_params:
      model: openai/gpt-4o
      api_key: os.environ/OPENAI_API_KEY

  - model_name: llama3.2             # self-hosted Ollama endpoint
    litellm_params:
      model: ollama/llama3.2
      api_base: http://ollama-service:11434

router_settings:
  fallbacks:                         # automatic failover
    - {gpt-4o-mini: [llama3.2]}
  retry_after: 5
  num_retries: 3

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
  database_url: os.environ/DATABASE_URL  # request logs
  max_budget: 50.0                   # USD/month hard cap across all keys
```

With the proxy running, we can point our `LLMClient` at it by changing only the base URL:

In [ ]:
import openai

# When LITELLM_PROXY_URL is set in the environment, all requests route
# through the proxy; otherwise fall back to the OpenAI endpoint directly.
PROXY_URL = os.getenv("LITELLM_PROXY_URL")  # e.g. "http://localhost:4000"

client_kwargs = {"api_key": os.getenv("OPENAI_API_KEY")}
if PROXY_URL:
    client_kwargs["base_url"] = f"{PROXY_URL}/v1"  # <1>

proxied_client = openai.OpenAI(**client_kwargs)

# Quick sanity check — model name routes through proxy alias:
try:
    resp = proxied_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "reply with: ok"}],
        max_tokens=5,
    )
    print(resp.choices[0].message.content)
except Exception as e:
    print(f"proxy not running locally — skipping: {e}")

1. `base_url` overrides the default `https://api.openai.com/v1` — the rest of the client API is identical, so no other changes are needed anywhere in the application.

## vLLM Local Inference

**Why vLLM?** When you self-host an open-weight model (Llama 3, Mistral, Phi-3, etc.) the naïve approach is to call `model.generate()` in a loop — one request at a time, one KV-cache allocation per request. Under any real concurrency this is catastrophically slow: each new request waits for the previous one to finish before its tokens are even scheduled on the GPU.

vLLM solves this with two techniques. **PagedAttention** partitions the GPU's KV-cache into fixed-size pages (analogous to virtual memory), allowing many requests to share the same physical memory without pre-allocating a contiguous block for each sequence's maximum length. **Continuous batching** (also called iteration-level scheduling) retires finished sequences mid-flight and slots new requests into the batch at the token level rather than waiting for the longest sequence to finish. Together these deliver throughputs 10–24× higher than HuggingFace `generate()` at the same hardware cost.

<br>

**OpenAI-compatible server.** vLLM ships a FastAPI server that exposes `POST /v1/chat/completions` with the same wire format as OpenAI. Any code that already calls `openai.chat.completions.create(...)` works unchanged — you only swap the `base_url`. This makes it a natural drop-in for the LiteLLM proxy config: add one entry in `model_list` and all application code gains access to the self-hosted model under an alias of your choosing.

<br>

**Hardware requirements.** vLLM requires CUDA and runs on any NVIDIA GPU with at least 8 GB VRAM for 7B-parameter models at 4-bit quantisation (`bitsandbytes`), or 16 GB for full-precision 7B. Multi-GPU tensor parallelism (`--tensor-parallel-size N`) scales to larger models across N GPUs. For CPU-only or Apple Silicon environments, use Ollama (already shown in the LiteLLM config above) as the local inference backend instead.

Launching a vLLM server for `meta-llama/Llama-3.2-3B-Instruct` on one GPU:

```bash
# Install (requires CUDA 12+ and Python 3.10+)
pip install vllm

# Start the OpenAI-compatible server
python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-3.2-3B-Instruct \  # <1>
    --dtype bfloat16 \                           # <2>
    --max-model-len 8192 \                       # <3>
    --port 8001 \                                # <4>
    --api-key local-secret                       # <5>
```

1. Any HuggingFace model ID or local path; the weights are downloaded to `~/.cache/huggingface` on first run.
2. `bfloat16` is the standard precision for Ampere+ GPUs (A100, 4090); use `float16` for Turing (T4, 3090).
3. Caps the maximum sequence length to reduce KV-cache memory pressure; set lower if you see OOM errors.
4. Any free port; we use `8001` to avoid conflicting with the FastAPI service on `8000`.
5. A static bearer token checked on every request; mount it from a Kubernetes `Secret` in production.

With the server running, the standard `openai` client connects by overriding `base_url`:

In [ ]:
import openai

VLLM_URL = os.getenv("VLLM_URL", "http://localhost:8001")
VLLM_KEY = os.getenv("VLLM_API_KEY", "local-secret")

vllm_client = openai.OpenAI(
    base_url=f"{VLLM_URL}/v1",  # <1>
    api_key=VLLM_KEY,
)

try:
    resp = vllm_client.chat.completions.create(
        model="meta-llama/Llama-3.2-3B-Instruct",  # <2>
        messages=[
            {"role": "system", "content": "You are a concise financial analyst."},
            {"role": "user",   "content": "What is PagedAttention in one sentence?"},
        ],
        max_tokens=80,
        temperature=0.0,
    )
    print(resp.choices[0].message.content)
except Exception as e:
    print(f"vLLM server not running locally — skipping: {e}")

1. The path `/v1` is mandatory — vLLM mounts all OpenAI-compatible endpoints under this prefix.
2. Pass the exact model ID that was given to `--model` at launch; vLLM returns a `404` for unknown model names.

To expose the vLLM backend through LiteLLM alongside the existing OpenAI entries, add one stanza to `litellm_config.yaml`:

```yaml
model_list:
  # ... existing OpenAI entries ...

  - model_name: llama3-local          # alias used by application code
    litellm_params:
      model: openai/meta-llama/Llama-3.2-3B-Instruct  # <1>
      api_base: http://vllm-service:8001               # <2>
      api_key: os.environ/VLLM_API_KEY

router_settings:
  fallbacks:
    - {gpt-4o-mini: [llama3-local]}    # <3>
```

1. The `openai/` prefix tells LiteLLM to use the OpenAI wire protocol against the custom `api_base`; the suffix is the model ID returned by the vLLM server's `/v1/models` endpoint.
2. In Kubernetes, `vllm-service` is a `ClusterIP` `Service` fronting the vLLM `Deployment`. GPU pods are typically scheduled on a dedicated node pool with a `nodeSelector` or `tolerations` for the GPU taint.
3. Listing `llama3-local` as a fallback for `gpt-4o-mini` means that if OpenAI is rate-limited or degraded, LiteLLM automatically retries against the self-hosted model — no code changes required in the application.

For containerised deployments, the official vLLM Docker image handles CUDA dependencies:

```dockerfile
# Deploy vLLM as a sidecar or standalone Deployment
FROM vllm/vllm-openai:latest

ENV HF_HOME=/model-cache
VOLUME ["/model-cache"]

CMD ["--model", "meta-llama/Llama-3.2-3B-Instruct",
     "--dtype", "bfloat16",
     "--max-model-len", "8192",
     "--port", "8001"]
```

:::{.callout-note}
Mount the model cache directory as a `PersistentVolume` in Kubernetes so weights are not re-downloaded on every pod restart. For large models (>30 GB) use a `ReadOnlyMany` PVC so multiple replicas share the same cached weights without duplicating storage cost.

:::

## Benchmarking

**What to measure.** Three metrics matter for an LLM service: (1) **time-to-first-token (TTFT)** — the latency a user perceives before they see any output, dominated by network RTT plus queuing at the provider; (2) **tokens-per-second (TPS)** — the streaming throughput once generation begins; (3) **end-to-end latency (P50/P95/P99)** — the full wall-clock time until the complete response arrives, relevant for synchronous API callers. For the filing analyser (a batch/API use case, not a streaming chat UI), end-to-end latency is the primary SLO metric.

<br>

**Concurrency matters.** A single sequential benchmark measures only serial latency — it does not reveal how the service behaves under load. We run $N$ requests concurrently using `asyncio.gather` to simulate real traffic. At high concurrency the bottleneck shifts from OpenAI API latency to our application's worker count, connection pool size, and the provider's rate-limit tokens-per-minute bucket.

<br>

**Interpreting the results.** P99 latency is typically 3–5× the median for LLM services because occasional slow completions and rate-limit retries have fat tails. An SLO of P99 < 10 s is a reasonable target for a filing analysis service where the caller is another service (not a human waiting for a chat response). If P99 exceeds this, the fix is usually (a) reducing `max_tokens`, (b) adding a circuit breaker that fails fast under load, or (c) upgrading to a faster model tier.

Defining the async benchmark harness:

In [ ]:
import asyncio, time
import openai
from dataclasses import dataclass, field

SAMPLE_TEXT = (
    "Net revenues for the fiscal year were $47.4 billion, an increase of 8% "
    "compared to the prior year. Our CET1 capital ratio was 14.8% at year-end, "
    "well above the 4.5% regulatory minimum. Interest rate risk is our most "
    "significant market risk; a 100 bps increase would reduce our fixed-rate "
    "portfolio fair value by $2.3 billion."
)

@dataclass
class BenchResult:
    latencies: list[float] = field(default_factory=list)

    def report(self):
        arr = np.sort(self.latencies)
        n   = len(arr)
        print(f"  n={n}  p50={arr[int(0.5*n)]:.2f}s  "
              f"p95={arr[int(0.95*n)]:.2f}s  p99={arr[min(int(0.99*n), n-1)]:.2f}s")

async def _single_request(aclient: openai.AsyncOpenAI) -> float:
    t0 = time.perf_counter()
    await aclient.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Return a JSON with keys summary (str) and risk_factors (list[str])."},
            {"role": "user",   "content": SAMPLE_TEXT},
        ],
        response_format={"type": "json_object"},
        max_tokens=256,
        temperature=0.0,
    )
    return time.perf_counter() - t0

async def benchmark(concurrency: int = 5) -> BenchResult:  # <1>
    aclient = openai.AsyncOpenAI()
    result  = BenchResult()
    tasks   = [_single_request(aclient) for _ in range(concurrency)]
    latencies = await asyncio.gather(*tasks, return_exceptions=True)  # <2>
    for lat in latencies:
        if isinstance(lat, float):
            result.latencies.append(lat)
    await aclient.close()
    return result

1. `concurrency` controls how many requests are fired simultaneously in a single `gather` batch — this simulates a burst of concurrent API callers.
2. `return_exceptions=True` prevents one failed request from cancelling the entire batch; exceptions are collected alongside timings and filtered out before reporting.

Running the benchmark at concurrency 1, 5, and 10:

In [ ]:
#| code-fold: true
async def run_all_benchmarks():
    for c in [1, 5, 10]:
        print(f"concurrency={c}")
        # Run three batches and pool results for stable percentiles
        all_lat = []
        for _ in range(3):
            r = await benchmark(concurrency=c)
            all_lat.extend(r.latencies)
        BenchResult(latencies=all_lat).report()

await run_all_benchmarks()

## Graceful Shutdown

**Why graceful shutdown matters.** Kubernetes sends `SIGTERM` to a pod before terminating it during a rolling deploy or scale-down event. Without a graceful shutdown handler, in-flight requests are dropped mid-response — callers receive a connection reset and must retry. With a handler, the server stops accepting new connections but allows active requests up to a timeout to complete before exiting. This is the single most impactful reliability improvement for an LLM service that has non-trivial per-request latency.

<br>

**Uvicorn's built-in support.** When launched via `uvicorn src.main:app --timeout-graceful-shutdown 30`, uvicorn catches `SIGTERM`, stops the accept loop, and waits up to 30 seconds for active connections to drain before sending `SIGKILL` to any remaining workers. This requires no application code changes — only the correct `CMD` in the Dockerfile.

<br>

**Pre-stop hook (Kubernetes).** Kubernetes also supports a `preStop` lifecycle hook that runs before `SIGTERM` is sent. We use a short `sleep` in this hook to give the load balancer time to drain traffic from the pod's endpoint before it starts refusing connections — without this, the load balancer may still route to the pod for a second or two after `SIGTERM` arrives.

The Kubernetes `Deployment` fragment with graceful shutdown configured:

```yaml
# k8s/deployment.yaml (relevant excerpts)
spec:
  template:
    spec:
      terminationGracePeriodSeconds: 60   # <1>
      containers:
        - name: filing-analyser
          image: myrepo/filing-analyser:latest
          ports:
            - containerPort: 8000
          lifecycle:
            preStop:
              exec:
                command: ["sleep", "15"]  # <2>
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 10
          env:
            - name: OPENAI_API_KEY
              valueFrom:
                secretKeyRef:
                  name: llm-secrets
                  key: openai-api-key
```

1. `terminationGracePeriodSeconds: 60` must be greater than the sum of the `preStop` sleep (15 s) and uvicorn's `--timeout-graceful-shutdown` (30 s), leaving a 15 s buffer for the process to exit cleanly.
2. The `preStop` sleep gives Kubernetes' endpoint controller time to remove the pod from the load balancer's active pool before `SIGTERM` is sent and uvicorn starts draining.

:::{.callout-caution}
The `readinessProbe` must point at `/health` and have a generous `initialDelaySeconds`. LLM services often have a slow first-request penalty (loading model configs, establishing connection pools). If the readiness probe fires before the service is ready, Kubernetes will repeatedly restart a healthy pod.

:::

## Exercises

1. **Add request-level logging middleware.** Implement a FastAPI middleware that logs the `request_id` (from an `X-Request-ID` header, or generated with `uuid4`), endpoint path, status code, and wall-clock latency for every request. Emit the log as a JSON line so it can be parsed by a log aggregator. Verify by hitting `POST /analyse` twice and checking that each log line has a unique `request_id`.

2. **Add a rate-limit header.** Extend the `/analyse` endpoint to read the `x-ratelimit-remaining-requests` header from the OpenAI response and attach it as a custom `X-Upstream-Remaining` header on the FastAPI response. This surfaces upstream quota exhaustion to the caller without them needing to parse OpenAI error bodies.

3. **Simulate a rolling deploy.** Write a script that (a) sends 20 concurrent requests to a running uvicorn server, (b) sends `SIGTERM` to the server process mid-flight using `os.kill(pid, signal.SIGTERM)`, and (c) counts how many of the 20 requests received a complete response vs. a connection error. The target is zero dropped requests with `--timeout-graceful-shutdown 30`.

---

$\blacksquare$